# 2. Demo - Spacecraft systematics

Learning goals:
- General setup of the PLATO payload
- Configuring systematic noise
- Ageing effects
- The PLATO PSF
- Random seeds

### Setup notebook

In [20]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figures in notebook
%matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Imports

In [21]:
import os
import numpy as np
import matplotlib.pyplot as plt

# PlatoSim
import platosim.utilities as ut
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

### Define variables

In [22]:
homeDir = os.getenv('PLATO_PROJECT_HOME')
workDir = os.getenv('PLATO_WORKDIR')

## Exercise 2.1 - The PLATO payload  

As a start we here take a look at how PlatoSim configures the spacecraft and generally sets up a simulation. As a top-down overview PlatoSim models the multi-camera payload configuration as illustrated in the figure below:

<img src="SpacecraftColor.png" width="600"/>

The sky pointing for PLATO is linked to the platform pointing (and not the spacecraft poiting as typically used for other spacecrafts). The default pointing of the platform (alpha, delta, kappa) corresponds to the PLATO LOPS1 field. 

Let's try to alter the pointing of the platform:

- <font color='orange'>Start by creating a new project folder called `demo1` placed in your PlatoSim working directory (**hint**: use the function `sim.createDirectory()`)</font>
- <font color='orange'>Setup a simulation where you change the solar panel orientation to 90 degress and run the simulation</font>
- <font color='orange'>Check the solar panel orientation in the `Platform` group within the HDF5 output file (e.g. using the function `h5.h5ls`)</font>

From the above exercise you may have noticed that you can also use the unit quaternion to transform from the equatorial to the platfrom reference frame. Generally, this method is mainly used by the PLATO Performance team (for now).

Like any other space telescopes placed in a L2 orbit, and that wants to monitor a patch of sky for an extended period, PLATO needs to make a 90 degree rotation approximately every 91 days (i.e. each mission quarter) in order to re-align its solar panels towards the Sun. Hence, the platform rotates along its roll axis with an angle of kappa = 90 deg. A realistic simulation needs to be broken up into time series of different mission quarters, where the solar panel orientation is set manually:
- <font color='orange'>Use your setup from above and change of the number of exposures to 1 exposure</font>
- <font color='orange'>Place your code into a small function that takes the solar panel as argument (as a float), place the execution call `sim.run()` in a for-loop for each corresponding solar panel orientation, and return the pixel image of each simulation</font>
- <font color='orange'>Test that your function reproduces the above simulation for using a solar panel orientation of 90 degress</font>
- <font color='orange'>Visualise the pixel image using `SimFile.showImage()` for the solar panel orientations [0, 90, 180, 270] degress</font>

Due to the opening angle for each of the four camera groups, the platform pointing is not identical to the pointing of the cameras. The camera pointing (i.e. the optical axis) can be derived from the platform pointing, and the tilt and azimuth angle. It is possible to figure the pointing of the Camera using the `Telescope` block (note *Telescope* is PlatoSim's internal naming convention).
- <font color='orange'>Inspect the the group `Telescope` within the YAML file</font>
- <font color='orange'>Setup a new simulation, set the parameter `GroupID = 1`, and run the simulation</font>
- <font color='orange'>Extract the pointing of the camera (which currently is identical to the camera-group)</font>

 <font color='red'>Report issue about wrong azimuth and tilt angles reported in the HDF5 file</font>

In case `GroupID = Custom`, the values for the tilt and azimuth angle of the camera-group will be read from the `Telescope` block.  In case of a pre-defined camera-group (i.e. `GroupID = 1, 2, 3, 4, or Fast`), the appropriate values will automatically be selected from the `CameraGroups` block:

    CameraGroups:                        # Do not alter these!
    
        AzimuthAngle:                    [45.0, 135.0, 225.0, 315.0, 0.0]
        TiltAngle:                       [9.2, 9.2, 9.2, 9.2, 0.0]
        
Note that **you should not alter any values from the CameraGroup block** unless you really attend to setup PlatoSim for a different payload configuration!

PlatoSim uses several reference frames to model incident light from the sky through the camera and on to the focal plane. As an overview we here show the CCD focal plane with the difference reference frames used internally by PlatoSim:

<img src="FocalPlaneCoordinateSystem.png" width="750"/>

More information about the sets of reference frame transformations is given in the PlatoSim paper. For now, what's important to know, is how to select a custom CCD position for your subfield:

- <font color='orange'>Inspect the `CCD` group within the YAML file</font>
- <font color='orange'>Use the simulation object from the previous exercise, select CCD #4, and run a new simulation</font>
- <font color='orange'>Check that your subfield is indeed on CCD #4 and not on CCD #1</font>
- <font color='orange'>Check visually that the subfield is different</font>

In case CCD `Position = Custom`, the values for the CCD origin offset, orientation, dimensions, and first illuminated row will be read from the `CCD` block.  In case of a pre-defined CCD position (i.e. `Position = 1, 2, 3, or 4`), the appropriate values will be selected from the `CCDPositions` block:

    CCDPositions:                        # Do not alter these!
    
        OriginOffsetX:                   [-1.3, -1.3, -1.3, -1.3]
        OriginOffsetY:                   [82.48, 82.48, 82.48, 82.48]
        Orientation:                     [180, 270, 0, 90] 
        NumColumns:                      [4510, 4510, 4510, 4510]
        NumRows:                         [4510, 4510, 4510, 4510]
        FirstRowForNormalCamera:         [0, 0, 0, 0]
        FirstRowForFastCamera:           [2255, 2255, 2255, 2255]

Again, note that **you should not alter any value from the CCDPositions block** unless you really attend to setup PlatoSim for a different payload configuration!

## Exercise 2.2 - Systematic noise

PlatoSim allows you to include several instrumental (systematic) noise sources, some of which are:

- AOCS jitter
- Thermal-Elastic Distortion (TED) of the camera
- Kinematic aberration
- Gain vs. temperature variations
- PSF breathing
- etc.

In the following we will focus on the built-in red noise model for the spacecraft jitter (ACS) and camera drift (TED).

Depending on the exact purpose of your simulations, you may want to alter the jitter and/or TED model of the simulated camera. Pointing variations of the payload (i.e. the platform and the cameras), are described in terms of Euler angles (yaw, pitch, roll) illustrated in the figure below.

<img src="PayloadReferenceFrame.png" width="600"/>

Since it is an art in itself to generate a realistic spacecraft jitter simulation, PlatoSim provide an alternative using a **Red Noise** (or "random walk") model. Such a model it also available for the TED drift. The Jitter and Drift parameters are the following:

- standard deviation ($\sigma$) of normal distribution describing (yaw, pitch, roll) step
- timescale ($\tau$)

The noise will be generated in the time domain with the following power spectral density:

$$P(\nu) = \frac{\sigma^2 \tau}{1 + (2\pi\nu\tau)^2} .$$

Highly realistic simulations of the jitter noise component typically means that a small time step needs to be considered (like 0.125s or 8 Hz). However, we need to be a bit pragmatic if the goal is to simulate a full mission quarter. The acutal time scale used internally within PlatoSim is 1/10th of the jitter time scale (we call this **PlatoSim's heartbeat**), hence, we recommend to never use a jitter time scale larger than 10 times the cadence (i.e. 250s for the normal N-CAM cadence). Otherwise, discontinuities may be observed in the simulated time series.

- <font color='orange'>Inspect the `Platform` group in the YAML file</font>
- <font color='orange'>Setup and run a simulation with red noise jitter (yaw, pitch, roll) = (0.04, 0.04, 0.04) arcsec</font>
- <font color='orange'>Setup and run a simulation with red noise jitter (yaw, pitch, roll) = (0.15, 0.15, 0.15) arcsec</font>
- <font color='orange'>Extract the platform coordinates and visually compare the yaw angle of the two simulations (**hint:** use the function `f.getPlatformYawPitchRoll()`)</font>

## Exercise 2.3 - Ageing effects

The `MissionDuration` is defined from the Beginning Of Life (BOL) to End Of Life (BOL) for the nominal mission. This parameter is key for time dependent effects that degrades the photometric quality - so-called ageing effects. The two most obvious effect are: 

- Decreasing transmission efficiency
- Charge Transfer Inefficiency (CTI)

Both of these effects are mainly caused by the constant collision of cosmic particles with the optics and detectors, respectively. In the following we will have a look at the impact of CTI on your simulations:

- <font color='orange'>Inspect the `CCD/CTI` subgroup of the YAML file</font>
- <font color='orange'>Setup and run a simulation of 1 exposure at BOL and EOL (**hint:** inspect the `ObservingParameters` group of the YAML file)</font>
- <font color='orange'>Select a subfield position at (row, column) = (3000, 1000)</font>
- <font color='orange'>Make a visual comparison between the two subfield to see the impact of CTI</font>

## Exercise 2.4 - The PLATO PSF

Below we show the non-diffused PSF grids across the FPA that are used by default in PlatoSim:

### Zemax PSF
<img src="psf_N6000K_Zemax.png" width="700"/>

### Analytic PSF
<img src="psf_N6000K_Analytical.png" width="700"/>

If case you want to use pre-computed normalised PSFs, you must provide these in the form of an HDF5-file. The path of this file, relative to the project location, is specified via the `MappedFromFile: Filename` parameter in the PSF block in the configuration file:

    PSF:

        Model:                       AnalyticNonGaussian      # PSF model
        MappedFromFile:                                       # Pre-computed PSF
            Filename:                inputfiles/PSF_Focus_0mu.hdf5
            NumberOfPixels:          8                        # Field size [pixel]
            ChargeDiffusionStrength: 0.2                      # [pixel]
            IncludeChargeDiffusion:  yes                      # [yes or no]
            IncludeJitterSmoothing:  no                       # [yes or no]
        AnalyticNonGaussian:
            ParameterFileName:       inputfiles/apsf_N6000K_v2.txt
            ChargeDiffusionStrength: 0.2                      # Width [pixels]
            IncludeChargeDiffusion:  yes                      # [yes or no]
            Sigma:                                            # Width [pixel]
                Source:              ConstantValue            # or FromFile
                ConstantValue:       0.5                      # [pixel]
                FromFile:            inputfiles/sigmaPSF.txt  # time & sigma
            
            
By default the analytic non-Gaussian PSF (`Model = AnalyticGaussian` in the PSF block), which is a best-fit model to the Zemax PSFs using an analytic approximation. The force of using the analytic model is the computational speed while model deviations are typically only around 2% (and maximally 5%). The most recent values for these parameters can be found in the file `apsf_N6000K_v2.txt` file in the `inputfiles/` directory. The simulator will automatically select the PSF for which the focal plane coordinates matches best the simulated subfield.

You can use the function `downloadFromFTP()` to download PSF files (or any other file) from the PlatoSim FTP server. E.g. say you wanted to download a Zemax PSF, then simply use:
```
path = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
ut.downloadFromFTP(filename="PSF_Focus_0mu.hdf5",  outputDir=path, server='plato')
```
A PSF HDF5 file is 6.8 Gb and hence it takes some time to download. To speed up things, among the files provided for this demo you should have received a compressed version of the file `PSF_Focus_0mu.hdf5`. 

- <font color='orange'>Start by decompressing (i.e. `unzip`) the PSF HDF5 file and place it in your project directory</font>
- <font color='orange'>Run a simulation with the new mapped PSF included (remember to save the high resolution PSF to the HDF5 output)</font>
- <font color='orange'>Compare the PSFs saved to the HDF5 file using the code snippet: `fig, ax = f.showPSF("highResPSF")`</font>

## Random seeds

The random seed below are used by the different noise generators of PlatoSim. Selecting proper "seeds" for each of these are very important if you want to micmic the future observations of PLATO:

    RandomSeeds:    
    
        ReadOutNoiseSeed:            1424949740
        PhotonNoiseSeed:             1433320336
        JitterSeed:                  1433320381
        FlatFieldSeed:               1425284070
        DriftSeed:                   1433429158
        CosmicSeed:                  1494750830
        DarkSignalSeed:              1468838669


If you are simulating a smaller subfield (like an imagette) for a duration of up to a full mission quarter, in order to simulate one star at a time, then you need to take care of all the seeds:

 - For each star: the seeds `ReadoutNoiseSeed`, `FlatFieldSeed`, and `DarkSignalSeed` need to be the different for all simulations, since the random noise fluctuation varies spatially across the FPA.
 - For each star: the `PhotonNoiseSeed` and `CosmicSeed` need to be different for all simulations since the arrival of incident light follows a Poisson distibution and varies spatially across the FPA.
 - For each camera: the `DriftSeed` needs to be the same for all stars observed with the same camera, but otherwise different.
 - For all simulations: the `JitterSeed` needs to be the same for all simulations covering the same time span since the jitter is identical for all stars and only varies over time.

<font color='orange'>**Extra exercise:**</font> 

- <font color='orange'>Setup a function that uses the above information and selects the corresponding seed for your observation (given the knowledge of which camera-group, camera, and CCD you are using).</font>
- <font color='orange'>We note that a red noise model is not very representative for the TED, hence a simple polynomial model can be generated and used as input instead. Create a polynomial model and include it using your `sim` object.</font> 